# E-commerce Sales Data Processing - pipeline

Runs the full pipeline: **unit tests -> raw tables -> enriched customers/products -> enriched orders -> profit aggregate -> reconciliation -> SQL reports**.

* Transformation logic: `scripts/transformations.py` (PySpark only)
* Orchestration and reconciliation: `scripts/pipeline.py`
* Tests: `tests/test_transformations.py`
* Design, data dictionary and data quality rules: `docs/`

Set the widgets at the top, then **Run all**.

In [0]:
import os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

dbutils.widgets.text("catalog", "dev_test", "Catalog")
dbutils.widgets.text("schema", "ecommerce_sales", "Schema")
# Relative paths are resolved from this notebook's folder; absolute paths (e.g. /Volumes/...) are used as-is
dbutils.widgets.text("data_path", "../data", "Source data folder")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
DATA_PATH_INPUT = dbutils.widgets.get("data_path")
DATA_PATH = (DATA_PATH_INPUT if os.path.isabs(DATA_PATH_INPUT)
             else os.path.abspath(os.path.join(os.getcwd(), DATA_PATH_INPUT)))

# Fail fast with a clear message if the source files are not where we expect them
REQUIRED_FILES = ["Orders.json", "Customer.xlsx", "Products.csv"]
missing = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(DATA_PATH, f))]
if missing and DATA_PATH.startswith("/Workspace/"):
    raise FileNotFoundError(
        f"Missing {missing} in data_path '{DATA_PATH_INPUT}'. Copy the source files into the repo "
        f"data/ folder or set the 'data_path' widget to the folder that contains them.")

# Print the widget value, not the absolute path (which contains the workspace user name)
print(f"catalog={CATALOG}  schema={SCHEMA}  data_path={DATA_PATH_INPUT}")

catalog=dev_test  schema=ecommerce_sales  data_path=../data


In [0]:
%run ./02_run_unit_tests

============================= test session starts ==============================
collecting ... collected 29 items

../tests/test_transformations.py::test_to_snake_case[Sub-Category-sub_category] PASSED [  3%]
../tests/test_transformations.py::test_to_snake_case[Customer ID-customer_id] PASSED [  6%]
../tests/test_transformations.py::test_to_snake_case[ Price per product -price_per_product] PASSED [ 10%]
../tests/test_transformations.py::test_to_snake_case[_corrupt_record-_corrupt_record] PASSED [ 13%]
../tests/test_transformations.py::test_require_columns_raises PASSED     [ 17%]
../tests/test_transformations.py::test_trim_strings_blank_to_null PASSED [ 20%]
../tests/test_transformations.py::test_customer_name_cleaning[Gary567 Hansen-Gary Hansen] PASSED [ 24%]
../tests/test_transformations.py::test_customer_name_cleaning[Tam&^*ara Willing___)ham-Tamara Willingham] PASSED [ 27%]
../tests/test_transformations.py::test_customer_name_cleaning[Ad.       ..am Hart-Adam Hart] PASSED [ 31%]
.

In [0]:
import importlib
import logging
import sys

sys.dont_write_bytecode = True
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
logging.basicConfig(level=logging.INFO)

from scripts import transformations, pipeline
importlib.reload(transformations)   # pick up code edits without restarting Python
importlib.reload(pipeline)
from scripts.pipeline import PipelineConfig, run_pipeline

cfg = PipelineConfig(catalog=CATALOG, schema=SCHEMA, data_path=DATA_PATH)

In [0]:
summary = run_pipeline(spark, cfg)
for k, v in summary.items():
    print(f"{k:30s} {v}")

INFO:ecommerce_pipeline:Wrote dev_test.ecommerce_sales.raw_orders
INFO:ecommerce_pipeline:Wrote dev_test.ecommerce_sales.raw_customers
INFO:ecommerce_pipeline:Wrote dev_test.ecommerce_sales.raw_products
INFO:ecommerce_pipeline:Wrote dev_test.ecommerce_sales.enriched_customers
INFO:ecommerce_pipeline:Wrote dev_test.ecommerce_sales.enriched_products
INFO:ecommerce_pipeline:Wrote dev_test.ecommerce_sales.enriched_orders
INFO:ecommerce_pipeline:Wrote dev_test.ecommerce_sales.quarantine_orders
INFO:ecommerce_pipeline:Wrote dev_test.ecommerce_sales.agg_profit
INFO:ecommerce_pipeline:Pipeline finished: {'raw_orders': 9994, 'enriched_orders': 9994, 'quarantined_orders': 0, 'orders_unknown_product': 204, 'orders_unknown_customer_name': 137, 'total_profit': Decimal('278417.03')}


raw_orders                     9994
enriched_orders                9994
quarantined_orders             0
orders_unknown_product         204
orders_unknown_customer_name   137
total_profit                   278417.03


In [0]:
for t in ["enriched_customers", "enriched_products", "enriched_orders", "agg_profit"]:
    print(t)
    display(spark.table(cfg.table(t)).limit(10))

enriched_customers


customer_id,customer_name,email,phone,address,segment,country,city,state,postal_code,region
AA-10315,Alex Avila,josephrice131@gmail.com,680-261-2092,"91773 Miller Shoal\nDiaztown, FL 38841",Consumer,United States,Round Rock,Texas,78664,Central
AA-10375,Allen Armold,garymoore386@gmail.com,221.945.4191x8872,"6450 John Lodge\nTerriton, KY 95945",Consumer,United States,Atlanta,Georgia,30318,South
AA-10480,Andrew Allen,johnwalker944@gmail.com,388-142-4883x5370,"27265 Murray Island\nKevinfort, PA 63231",Consumer,United States,Concord,North Carolina,28027,South
AA-10645,Anna Andreadi,ericcarter176@gmail.com,(451)259-5402,USNS Knight\nFPO AA 76532,Consumer,United States,Chester,Pennsylvania,19013,East
AB-10015,Aaron Bergman,williamjackson427@gmail.com,625.626.9133x374,"170 Jackson Loaf\nKristenland, AS 48876",Consumer,United States,Arlington,Texas,76017,Central
AB-10060,Adam Bellavance,nicolejones228@gmail.com,1783534757,"14354 Baker Harbor Apt. 017\nEricville, HI 11192",Home Office,United States,Des Moines,Washington,98198,West
AB-10105,Adrian Barton,daviddavis980@gmail.com,067.435.8553x692,"021 Katherine Mall\nJameston, DC 24685",Consumer,United States,Phoenix,Arizona,85023,West
AB-10150,Aimee Bixby,jasonmontgomery889@gmail.com,001-208-220-1519,"14657 Scott Loop Apt. 735\nPort Ashley, NH 34470",Consumer,United States,Dallas,Texas,75220,Central
AB-10165,Alan Barnes,kentweaver695@gmail.com,(000)101-6979x011,"6644 Mitchell Burg\nVictorhaven, KS 66356",Consumer,United States,Toledo,Ohio,43615,East
AB-10255,Alejandro Ballentine,darrelldillon573@gmail.com,(156)423-0600x52870,"PSC 7267, Box 9911\nAPO AE 49343",Home Office,United States,Jackson,Mississippi,39212,South


enriched_products


product_id,category,sub_category,product_name,state,price_per_product
FUR-BO-10000112,Furniture,Bookcases,"Bush Birmingham Collection Bookcase, Dark Cherry",Illinois,91.6860
FUR-BO-10000330,Furniture,Bookcases,"Sauder Camden County Barrister Bookcase, Planked Cherry Finish",Louisiana,120.9800
FUR-BO-10000362,Furniture,Bookcases,Sauder Inglewood Library Bookcases,California,145.3330
FUR-BO-10000468,Furniture,Bookcases,O'Sullivan 2-Shelf Heavy-Duty Bookcases,Washington,48.5800
FUR-BO-10000711,Furniture,Bookcases,"Hon Metal Bookcases, Gray",Arkansas,70.9800
FUR-BO-10000780,Furniture,Bookcases,O'Sullivan Plantations 2-Door Library in Landvery Oak,Pennsylvania,100.4900
FUR-BO-10001337,Furniture,Bookcases,O'Sullivan Living Dimensions 2-Shelf Bookcases,California,102.8330
FUR-BO-10001519,Furniture,Bookcases,O'Sullivan 3-Shelf Heavy-Duty Bookcases,Virginia,58.1400
FUR-BO-10001567,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Dark Cherry Finish, Fully Assembled",Colorado,30.2940
FUR-BO-10001601,Furniture,Bookcases,"Sauder Mission Library with Doors, Fruitwood Finish",Texas,89.0664


enriched_orders


row_id,order_id,order_date,order_year,ship_date,ship_mode,customer_id,customer_name,country,product_id,category,sub_category,quantity,price,discount,profit
28,CA-2015-140025,2015-04-07,2015,2015-04-11,Standard Class,PF-19120,Peter Fuller,United States,OFF-AP-10002651,Office Supplies,Appliances,8,463.25,0.8,-1181.28
29,CA-2014-103331,2014-08-26,2014,2014-09-01,Standard Class,KB-16315,Karl Braun,United States,OFF-PA-10002659,Office Supplies,Paper,1,3.38,0.0,1.55
30,CA-2017-118003,2017-12-04,2017,2017-12-10,Standard Class,DO-13645,Doug O'Connell,United States,FUR-CH-10004860,Furniture,Chairs,3,302.94,0.0,18.18
33,CA-2015-123092,2015-12-03,2015,2015-12-07,Standard Class,JG-15115,Jack Garza,United States,OFF-BI-10001718,Office Supplies,Binders,5,95.00,0.7,-73.58
93,CA-2014-103401,2014-11-21,2014,2014-11-26,Standard Class,GR-14560,Georgia Rosenberg,United States,OFF-PA-10003309,Office Supplies,Paper,2,12.00,0.0,6.22
151,CA-2017-128853,2017-04-21,2017,2017-04-23,First Class,JM-15250,Janet Martin,United States,FUR-CH-10004218,Furniture,Chairs,9,908.82,0.0,227.21
171,CA-2017-127432,2017-01-22,2017,2017-01-27,Standard Class,AD-10180,Alan Dominguez,United States,TEC-CO-10003236,Technology,Copiers,5,2999.95,0.0,1379.98
200,CA-2016-161158,2016-05-09,2016,2016-05-13,Standard Class,PS-18970,Paul Stevenson,United States,OFF-AR-10000462,Office Supplies,Art,5,8.00,0.0,3.44
208,CA-2016-108581,2016-06-20,2016,2016-06-26,Standard Class,EA-14035,Erin Ashbrook,United States,TEC-AC-10001109,Technology,Accessories,4,95.00,0.2,26.39
221,CA-2016-133711,2016-11-26,2016,2016-11-29,First Class,MC-17425,Mark Cousins,United States,OFF-PA-10001685,Office Supplies,Paper,7,70.98,0.0,34.78


agg_profit


order_year,category,sub_category,customer_id,customer_name,total_profit,order_count,total_quantity
2017,Furniture,Bookcases,TS-21505,Unknown,-183.63,1,7
2017,Office Supplies,Storage,JO-15280,Jas O'Carroll,-54.88,1,5
2017,Technology,Phones,RD-19480,Rick Duston,75.41,2,12
2017,Office Supplies,Storage,PF-19165,Philip Fox,4.07,1,1
2017,Office Supplies,Paper,DH-13075,Dave Hallsten,28.80,1,9
2016,Technology,Phones,MY-18295,Muhammed YeDwab,0.26,1,1
2014,Technology,Phones,SB-20185,Sarah Brown,3.48,1,2
2015,Office Supplies,Storage,SP-20650,Stephanie Phelps,21.03,1,6
2017,Technology,Phones,MT-18070,Michelle Tran,17.50,1,5
2017,Technology,Phones,CC-12220,Chris Cortes,125.37,2,20


## Task 5: SQL aggregates
The queries read `agg_profit` from the catalog and schema set in the widgets.

In [0]:
%sql
SELECT order_year, SUM(total_profit) AS profit
FROM IDENTIFIER(:catalog || '.' || :schema || '.agg_profit')
GROUP BY order_year
ORDER BY order_year

order_year,profit
2014,39185.75
2015,63073.09
2016,65073.23
2017,111084.96


In [0]:
%sql
SELECT order_year, category, SUM(total_profit) AS profit
FROM IDENTIFIER(:catalog || '.' || :schema || '.agg_profit')
GROUP BY order_year, category
ORDER BY order_year, category

order_year,category,profit
2014,Furniture,-5331.04
2014,Office Supplies,22500.32
2014,Technology,21493.35
2014,Unknown,523.12
2015,Furniture,3027.18
2015,Office Supplies,24519.39
2015,Technology,34943.36
2015,Unknown,583.16
2016,Furniture,6889.50
2016,Office Supplies,34555.53


In [0]:
%sql
SELECT customer_id, customer_name, SUM(total_profit) AS profit
FROM IDENTIFIER(:catalog || '.' || :schema || '.agg_profit')
GROUP BY customer_id, customer_name
ORDER BY profit DESC

customer_id,customer_name,profit
FH-14275,Frank Hawley,13400.32
TC-20980,Tamara Chand,8981.32
RB-19360,Raymond Buch,6976.36
SC-20095,Sanjit Chand,5757.30
HL-15040,Hunter Lopez,5622.43
PR-18880,Patrick Ryan,5596.20
AB-10105,Adrian Barton,5444.97
TA-21385,Tom Ashbrook,4703.72
CM-12385,Christopher Martinez,3900.03
PS-19045,Penelope Sewall,3183.77


In [0]:
%sql
SELECT customer_id, customer_name, order_year, SUM(total_profit) AS profit
FROM IDENTIFIER(:catalog || '.' || :schema || '.agg_profit')
GROUP BY customer_id, customer_name, order_year
ORDER BY customer_id, order_year

customer_id,customer_name,order_year,profit
AA-10315,Alex Avila,2014,280.68
AA-10315,Alex Avila,2015,7.00
AA-10315,Alex Avila,2016,-747.09
AA-10315,Alex Avila,2017,96.50
AA-10375,Allen Armold,2014,16.72
AA-10375,Allen Armold,2015,69.03
AA-10375,Allen Armold,2016,122.73
AA-10375,Allen Armold,2017,68.92
AA-10480,Andrew Allen,2014,9.89
AA-10480,Andrew Allen,2016,420.75
